In [0]:
import dlt
from pyspark.sql.functions import col, year, month, dayofmonth, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [0]:
# Set schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

In [0]:
spark.conf.set("fs.azure.account.key.gksdatalake4.dfs.core.windows.net", "efnbj7n1uTmKrsGIOFTuOFlsvLMJSoxh8yUkJ8+WUC/0ZLk4Jdr6glal1HmqbiPg9zK7SGrM9mpe+AStL9Gkqg==")

In [0]:
# Step 1: Bronze Ingestion Table from ADLS (Auto Loader)
@dlt.table(
    name="ecomm_bronze",
    comment="Raw data loaded from ADLS Gen2 Bronze zone using Auto Loader",
)
def bronze_table():
    from pyspark.sql.functions import col, year, month, dayofmonth, to_timestamp
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

    ecomm_schema = StructType([
        StructField("InvoiceNo", StringType(), True),
        StructField("StockCode", StringType(), True),
        StructField("Description", StringType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("InvoiceDate", TimestampType(), True),
        StructField("UnitPrice", DoubleType(), True),
        StructField("CustomerID", IntegerType(), True),
        StructField("Country", StringType(), True)
   ])
    
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(ecomm_schema)
        .load("abfss://bronze@gksdatalake4.dfs.core.windows.net/ecomm/")
    )

In [0]:
# Step 2: Cleaned Silver Table
# TODO: add expectations, drops here
@dlt.table(
    name="ecomm_silver",
    comment="Cleaned e-commerce data with filters applied and Amount column calculated",
)
def silver_table():
    df = dlt.read_stream("ecomm_bronze")

    cleaned_df = (
        df
        .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss"))
        .withColumn("StockCode", col("StockCode").cast("string"))
        .filter(~col("InvoiceNo").cast("string").startswith("C"))
        .filter(col("CustomerID").isNotNull())
        .filter(col("Quantity") >= 0)
        .withColumn("year", year(col("InvoiceDate")))
        .withColumn("month", month(col("InvoiceDate")))
        .withColumn("day", dayofmonth(col("InvoiceDate")))
        .withColumn("Amount", col("Quantity") * col("UnitPrice"))
    )

    return cleaned_df

In [0]:
import dlt
from pyspark.sql.functions import col, sum as _sum, window

 

@dlt.table(
    name="country_hourly_sales2",
    comment="Hourly aggregated total sales per country from Silver data",
    table_properties={
        "delta.targetFileSize": "134217728",
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact":  "true"
    }
)
def country_hourly_sales():
    df = dlt.read_stream("ecomm_silver")

    aggregated = (
        df.groupBy(
            window("InvoiceDate", "1 hour"),
            col("Country")
        )
        .agg(
            _sum("Amount").alias("TotalSales")
        )
        .select(
            col("window.start").alias("HourStart"),
            col("window.end").alias("HourEnd"),
            col("Country"),
            col("TotalSales")
        )
    )

    return aggregated